# M1

In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. LOAD DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\train_M1.csv"
)

test_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\test_M1.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

selected_features = [
    'Mean_temp',
    'Max_temp',
    'Min_temp',
    'Diurnal_temp',
    'Rainfall_mean',
    'Relative_humidity',
    'Population_density',
    'Mean_temp_lag1',
    'Mean_temp_lag2',
    'Mean_temp_lag3',
    'Max_temp_lag1',
    'Max_temp_lag2',
    'Max_temp_lag3',
    'Min_temp_lag1',
    'Min_temp_lag2',
    'Min_temp_lag3',
    'Diurnal_temp_lag1',
    'Diurnal_temp_lag2',
    'Diurnal_temp_lag3',
    'Rainfall_mean_lag1',
    'Rainfall_mean_lag2',
    'Rainfall_mean_lag3',
    'Relative_humidity_lag1',
    'Relative_humidity_lag2',
    'Relative_humidity_lag3',
    'Cases_lag1'
]


# ============================================================
# 3. TRAIN / TEST DATA
# ============================================================

X_train = train_df[selected_features]
y_train = train_df['Cases']

X_test = test_df[selected_features]
y_test = test_df['Cases']


# ============================================================
# 4. PARAMETER GRID
# ============================================================

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.05, 0.1, 0.5],
    'subsample': [0.5, 1.0],
    'colsample_bytree': [0.5, 1.0]
}


# ============================================================
# 5. XGBOOST MODEL
# ============================================================

xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 6. GRID SEARCH
# ============================================================

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)


# ============================================================
# 7. GRID SEARCH RESULTS
# ============================================================

results_df = pd.DataFrame(grid.cv_results_)

print("\nTotal parameter combinations:")
print(len(results_df))


# ============================================================
# 8. FUNCTION FOR MAPE AND MPE
# ============================================================

def calculate_mape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove zero observed values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


def calculate_mpe(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove zero observed values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 9. CALCULATE TRAIN AND TEST METRICS
#    FOR EVERY PARAMETER COMBINATION
# ============================================================

metrics_list = []

for i, params in enumerate(results_df['params']):

    print(
        f"Processing combination {i + 1} "
        f"of {len(results_df)}"
    )

    # Create model using current parameters
    model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        **params
    )

    # Fit model
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # TRAIN PREDICTIONS
    # --------------------------------------------------------

    y_train_pred = model.predict(X_train)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, y_train_pred)
    )

    train_mae = mean_absolute_error(
        y_train,
        y_train_pred
    )

    train_mape = calculate_mape(
        y_train,
        y_train_pred
    )

    train_mpe = calculate_mpe(
        y_train,
        y_train_pred
    )

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )


    # --------------------------------------------------------
    # TEST PREDICTIONS
    # --------------------------------------------------------

    y_test_pred = model.predict(X_test)

    test_rmse = np.sqrt(
        mean_squared_error(y_test, y_test_pred)
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )

    test_mape = calculate_mape(
        y_test,
        y_test_pred
    )

    test_mpe = calculate_mpe(
        y_test,
        y_test_pred
    )

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )


    # --------------------------------------------------------
    # SAVE METRICS
    # --------------------------------------------------------

    metrics_list.append({

        # Parameter values
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'learning_rate': params['learning_rate'],
        'subsample': params['subsample'],
        'colsample_bytree': params['colsample_bytree'],

        # Train metrics
        'Train_RMSE': train_rmse,
        'Train_MAE': train_mae,
        'Train_MAPE (%)': train_mape,
        'Train_MPE (%)': train_mpe,
        'Train_R2': train_r2,

        # Test metrics
        'Test_RMSE': test_rmse,
        'Test_MAE': test_mae,
        'Test_MAPE (%)': test_mape,
        'Test_MPE (%)': test_mpe,
        'Test_R2': test_r2
    })


# ============================================================
# 10. CREATE METRICS DATAFRAME
# ============================================================

metrics_df = pd.DataFrame(metrics_list)


# ============================================================
# 11. ADD GRIDSEARCH CV RESULTS
# ============================================================

metrics_df['CV_RMSE'] = (
    -results_df['mean_test_score']
)

metrics_df['CV_RMSE_SD'] = (
    results_df['std_test_score']
)


# ============================================================
# 12. IDENTIFY BEST PARAMETER BASED ON CV RMSE
# ============================================================

metrics_df['Best_CV_Model'] = False

best_index = (
    metrics_df['CV_RMSE']
    .idxmin()
)

metrics_df.loc[
    best_index,
    'Best_CV_Model'
] = True


# ============================================================
# 13. SORT BY TEST RMSE
# ============================================================

metrics_df = metrics_df.sort_values(
    by='Test_RMSE',
    ascending=True
).reset_index(drop=True)


# ============================================================
# 14. SAVE ALL RESULTS TO EXCEL
# ============================================================

output_file = (
    r"C:\Users\HPC\Desktop\DR.Sreelakshmi"
    r"\ML Revision\xgboost_M1_all_combinations.xlsx"
)

metrics_df.to_excel(
    output_file,
    index=False,
    sheet_name='All_Results'
)


# ============================================================
# 15. PRINT RESULTS
# ============================================================

print("\n==========================================")
print("ALL RESULTS SAVED SUCCESSFULLY")
print("==========================================")

print(f"\nFile saved at:")
print(output_file)

print("\nBest parameter combination based on CV RMSE:")
print(
    metrics_df[
        metrics_df['Best_CV_Model'] == True
    ]
)

print("\nTop 10 combinations based on Test RMSE:")
print(
    metrics_df.head(10)
)

Train shape: (4060, 32)
Test shape: (1015, 32)
Fitting 5 folds for each of 240 candidates, totalling 1200 fits

Total parameter combinations:
240
Processing combination 1 of 240
Processing combination 2 of 240
Processing combination 3 of 240
Processing combination 4 of 240
Processing combination 5 of 240
Processing combination 6 of 240
Processing combination 7 of 240
Processing combination 8 of 240
Processing combination 9 of 240
Processing combination 10 of 240
Processing combination 11 of 240
Processing combination 12 of 240
Processing combination 13 of 240
Processing combination 14 of 240
Processing combination 15 of 240
Processing combination 16 of 240
Processing combination 17 of 240
Processing combination 18 of 240
Processing combination 19 of 240
Processing combination 20 of 240
Processing combination 21 of 240
Processing combination 22 of 240
Processing combination 23 of 240
Processing combination 24 of 240
Processing combination 25 of 240
Processing combination 26 of 240
Proce

In [6]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. LOAD DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\train_M1.csv"
)

test_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\test_M1.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

selected_features = [
    'Mean_temp',
    'Max_temp',
    'Min_temp',
    'Diurnal_temp',
    'Rainfall_mean',
    'Relative_humidity',
    'Population_density',
    'Mean_temp_lag1',
    'Mean_temp_lag2',
    'Mean_temp_lag3',
    'Max_temp_lag1',
    'Max_temp_lag2',
    'Max_temp_lag3',
    'Min_temp_lag1',
    'Min_temp_lag2',
    'Min_temp_lag3',
    'Diurnal_temp_lag1',
    'Diurnal_temp_lag2',
    'Diurnal_temp_lag3',
    'Rainfall_mean_lag1',
    'Rainfall_mean_lag2',
    'Rainfall_mean_lag3',
    'Relative_humidity_lag1',
    'Relative_humidity_lag2',
    'Relative_humidity_lag3',
    'Cases_lag1'
]


# ============================================================
# 3. TRAIN / TEST DATA
# ============================================================

X_train = train_df[selected_features]
y_train = train_df['Cases']

X_test = test_df[selected_features]
y_test = test_df['Cases']


# ============================================================
# 4. SELECTED XGBOOST PARAMETERS
# ============================================================

params = {
    'colsample_bytree': 1.0,
    'learning_rate': 0.05,
    'max_depth': 3,
    'n_estimators': 200,
    'subsample': 0.8
}


# ============================================================
# 5. BUILD XGBOOST MODEL
# ============================================================

model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    **params
)


# ============================================================
# 6. FIT MODEL
# ============================================================

model.fit(X_train, y_train)


# ============================================================
# 7. PREDICTIONS
# ============================================================

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


# ============================================================
# 8. MAPE FUNCTION
# ============================================================

def calculate_mape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exclude observed = 0
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 9. MPE FUNCTION
# ============================================================

def calculate_mpe(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exclude observed = 0
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 10. TRAIN METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(y_train, y_train_pred)
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

train_mape = calculate_mape(
    y_train,
    y_train_pred
)

train_mpe = calculate_mpe(
    y_train,
    y_train_pred
)

train_r2 = r2_score(
    y_train,
    y_train_pred
)


# ============================================================
# 11. TEST METRICS
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_mape = calculate_mape(
    y_test,
    y_test_pred
)

test_mpe = calculate_mpe(
    y_test,
    y_test_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


# ============================================================
# 12. PRINT SELECTED PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("SELECTED XGBOOST M1 PARAMETERS")
print("=" * 60)

for key, value in params.items():
    print(f"{key}: {value}")

print("random_state: 42")


# ============================================================
# 13. PRINT TRAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MAPE : {train_mape:.4f}%")
print(f"MPE  : {train_mpe:.4f}%")
print(f"R²   : {train_r2:.4f}")


# ============================================================
# 14. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MAPE : {test_mape:.4f}%")
print(f"MPE  : {test_mpe:.4f}%")
print(f"R²   : {test_r2:.4f}")

print("\n" + "=" * 60)
print("MODEL RUN COMPLETED")
print("=" * 60)

Train shape: (4060, 32)
Test shape: (1015, 32)

SELECTED XGBOOST M1 PARAMETERS
colsample_bytree: 1.0
learning_rate: 0.05
max_depth: 3
n_estimators: 200
subsample: 0.8
random_state: 42

TRAINING RESULTS
RMSE : 2.1005
MAE  : 0.9241
MAPE : 62.4954%
MPE  : 20.6188%
R²   : 0.6621

TEST RESULTS
RMSE : 3.7331
MAE  : 1.3521
MAPE : 61.9378%
MPE  : 19.1616%
R²   : 0.5495

MODEL RUN COMPLETED


# M2

In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. LOAD DATA
# ============================================================

train_df = pd.read_csv("C:\\Users\\HPC\\Downloads\\New_ML\\train_test_data\\train_M2.csv")
test_df = pd.read_csv("C:\\Users\\HPC\\Downloads\\New_ML\\train_test_data\\test_M2.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

selected_features = [
   'HI', 'BI', 'CI', 'HI_lag1', 'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2',
       'BI_lag3', 'CI_lag1', 'CI_lag2', 'CI_lag3', 'Cases_lag1', 'Population_density'
]


# ============================================================
# 3. TRAIN / TEST DATA
# ============================================================

X_train = train_df[selected_features]
y_train = train_df['Cases']

X_test = test_df[selected_features]
y_test = test_df['Cases']


# ============================================================
# 4. PARAMETER GRID
# ============================================================

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.05, 0.1, 0.5],
    'subsample': [0.5, 1.0],
    'colsample_bytree': [0.5, 1.0]
}


# ============================================================
# 5. XGBOOST MODEL
# ============================================================

xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 6. GRID SEARCH
# ============================================================

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)


# ============================================================
# 7. GRID SEARCH RESULTS
# ============================================================

results_df = pd.DataFrame(grid.cv_results_)

print("\nTotal parameter combinations:")
print(len(results_df))


# ============================================================
# 8. FUNCTION FOR MAPE AND MPE
# ============================================================

def calculate_mape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove zero observed values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


def calculate_mpe(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove zero observed values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 9. CALCULATE TRAIN AND TEST METRICS
#    FOR EVERY PARAMETER COMBINATION
# ============================================================

metrics_list = []

for i, params in enumerate(results_df['params']):

    print(
        f"Processing combination {i + 1} "
        f"of {len(results_df)}"
    )

    # Create model using current parameters
    model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        **params
    )

    # Fit model
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # TRAIN PREDICTIONS
    # --------------------------------------------------------

    y_train_pred = model.predict(X_train)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, y_train_pred)
    )

    train_mae = mean_absolute_error(
        y_train,
        y_train_pred
    )

    train_mape = calculate_mape(
        y_train,
        y_train_pred
    )

    train_mpe = calculate_mpe(
        y_train,
        y_train_pred
    )

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )


    # --------------------------------------------------------
    # TEST PREDICTIONS
    # --------------------------------------------------------

    y_test_pred = model.predict(X_test)

    test_rmse = np.sqrt(
        mean_squared_error(y_test, y_test_pred)
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )

    test_mape = calculate_mape(
        y_test,
        y_test_pred
    )

    test_mpe = calculate_mpe(
        y_test,
        y_test_pred
    )

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )


    # --------------------------------------------------------
    # SAVE METRICS
    # --------------------------------------------------------

    metrics_list.append({

        # Parameter values
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'learning_rate': params['learning_rate'],
        'subsample': params['subsample'],
        'colsample_bytree': params['colsample_bytree'],

        # Train metrics
        'Train_RMSE': train_rmse,
        'Train_MAE': train_mae,
        'Train_MAPE (%)': train_mape,
        'Train_MPE (%)': train_mpe,
        'Train_R2': train_r2,

        # Test metrics
        'Test_RMSE': test_rmse,
        'Test_MAE': test_mae,
        'Test_MAPE (%)': test_mape,
        'Test_MPE (%)': test_mpe,
        'Test_R2': test_r2
    })


# ============================================================
# 10. CREATE METRICS DATAFRAME
# ============================================================

metrics_df = pd.DataFrame(metrics_list)


# ============================================================
# 11. ADD GRIDSEARCH CV RESULTS
# ============================================================

metrics_df['CV_RMSE'] = (
    -results_df['mean_test_score']
)

metrics_df['CV_RMSE_SD'] = (
    results_df['std_test_score']
)


# ============================================================
# 12. IDENTIFY BEST PARAMETER BASED ON CV RMSE
# ============================================================

metrics_df['Best_CV_Model'] = False

best_index = (
    metrics_df['CV_RMSE']
    .idxmin()
)

metrics_df.loc[
    best_index,
    'Best_CV_Model'
] = True


# ============================================================
# 13. SORT BY TEST RMSE
# ============================================================

metrics_df = metrics_df.sort_values(
    by='Test_RMSE',
    ascending=True
).reset_index(drop=True)


# ============================================================
# 14. SAVE ALL RESULTS TO EXCEL
# ============================================================

output_file = (
    r"C:\Users\HPC\Desktop\DR.Sreelakshmi"
    r"\ML Revision\XGBoost\xgboost_M2_all_combinations.xlsx"
)

metrics_df.to_excel(
    output_file,
    index=False,
    sheet_name='All_Results'
)


# ============================================================
# 15. PRINT RESULTS
# ============================================================

print("\n==========================================")
print("ALL RESULTS SAVED SUCCESSFULLY")
print("==========================================")

print(f"\nFile saved at:")
print(output_file)

print("\nBest parameter combination based on CV RMSE:")
print(
    metrics_df[
        metrics_df['Best_CV_Model'] == True
    ]
)

print("\nTop 10 combinations based on Test RMSE:")
print(
    metrics_df.head(10)
)

Train shape: (4060, 20)
Test shape: (1015, 20)
Fitting 5 folds for each of 240 candidates, totalling 1200 fits

Total parameter combinations:
240
Processing combination 1 of 240
Processing combination 2 of 240
Processing combination 3 of 240
Processing combination 4 of 240
Processing combination 5 of 240
Processing combination 6 of 240
Processing combination 7 of 240
Processing combination 8 of 240
Processing combination 9 of 240
Processing combination 10 of 240
Processing combination 11 of 240
Processing combination 12 of 240
Processing combination 13 of 240
Processing combination 14 of 240
Processing combination 15 of 240
Processing combination 16 of 240
Processing combination 17 of 240
Processing combination 18 of 240
Processing combination 19 of 240
Processing combination 20 of 240
Processing combination 21 of 240
Processing combination 22 of 240
Processing combination 23 of 240
Processing combination 24 of 240
Processing combination 25 of 240
Processing combination 26 of 240
Proce

In [7]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. LOAD DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\train_M2.csv"
)

test_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\test_M2.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

selected_features = [
   'HI', 'BI', 'CI', 'HI_lag1', 'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2',
       'BI_lag3', 'CI_lag1', 'CI_lag2', 'CI_lag3', 'Cases_lag1', 'Population_density'
]


# ============================================================
# 3. TRAIN / TEST DATA
# ============================================================

X_train = train_df[selected_features]
y_train = train_df['Cases']

X_test = test_df[selected_features]
y_test = test_df['Cases']


# ============================================================
# 4. SELECTED XGBOOST PARAMETERS
# ============================================================

params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'max_depth': 3,
    'n_estimators': 100,
    'subsample': 0.8
}


# ============================================================
# 5. BUILD XGBOOST MODEL
# ============================================================

model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    **params
)


# ============================================================
# 6. FIT MODEL
# ============================================================

model.fit(X_train, y_train)


# ============================================================
# 7. PREDICTIONS
# ============================================================

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


# ============================================================
# 8. MAPE FUNCTION
# ============================================================

def calculate_mape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exclude observed = 0
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 9. MPE FUNCTION
# ============================================================

def calculate_mpe(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exclude observed = 0
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 10. TRAIN METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(y_train, y_train_pred)
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

train_mape = calculate_mape(
    y_train,
    y_train_pred
)

train_mpe = calculate_mpe(
    y_train,
    y_train_pred
)

train_r2 = r2_score(
    y_train,
    y_train_pred
)


# ============================================================
# 11. TEST METRICS
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_mape = calculate_mape(
    y_test,
    y_test_pred
)

test_mpe = calculate_mpe(
    y_test,
    y_test_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


# ============================================================
# 12. PRINT SELECTED PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("SELECTED XGBOOST M2 PARAMETERS")
print("=" * 60)

for key, value in params.items():
    print(f"{key}: {value}")

print("random_state: 42")


# ============================================================
# 13. PRINT TRAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MAPE : {train_mape:.4f}%")
print(f"MPE  : {train_mpe:.4f}%")
print(f"R²   : {train_r2:.4f}")


# ============================================================
# 14. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MAPE : {test_mape:.4f}%")
print(f"MPE  : {test_mpe:.4f}%")
print(f"R²   : {test_r2:.4f}")

print("\n" + "=" * 60)
print("MODEL RUN COMPLETED")
print("=" * 60)

Train shape: (4060, 20)
Test shape: (1015, 20)

SELECTED XGBOOST M2 PARAMETERS
colsample_bytree: 0.8
learning_rate: 0.05
max_depth: 3
n_estimators: 100
subsample: 0.8
random_state: 42

TRAINING RESULTS
RMSE : 2.4942
MAE  : 1.0850
MAPE : 69.7247%
MPE  : 24.5464%
R²   : 0.5235

TEST RESULTS
RMSE : 4.5452
MAE  : 1.6329
MAPE : 79.0567%
MPE  : 6.8085%
R²   : 0.3322

MODEL RUN COMPLETED


# M3

In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. LOAD DATA
# ============================================================

train_df = pd.read_csv("C:\\Users\\HPC\\Downloads\\New_ML\\train_test_data\\train_M3.csv")
test_df = pd.read_csv("C:\\Users\\HPC\\Downloads\\New_ML\\train_test_data\\test_M3.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

selected_features = [
    'HI', 'BI', 'CI', 'HI_lag1', 'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2',
       'BI_lag3', 'CI_lag1', 'CI_lag2', 'CI_lag3',
    'Mean_temp', 'Max_temp', 'Min_temp', 'Diurnal_temp', 'Rainfall_mean',
       'Relative_humidity', 'Population_density', 'Mean_temp_lag1',
       'Mean_temp_lag2', 'Mean_temp_lag3', 'Max_temp_lag1', 'Max_temp_lag2',
       'Max_temp_lag3', 'Min_temp_lag1', 'Min_temp_lag2', 'Min_temp_lag3',
       'Diurnal_temp_lag1', 'Diurnal_temp_lag2', 'Diurnal_temp_lag3',
       'Rainfall_mean_lag1', 'Rainfall_mean_lag2', 'Rainfall_mean_lag3',
       'Relative_humidity_lag1', 'Relative_humidity_lag2',
       'Relative_humidity_lag3', 'Cases_lag1'
]


# ============================================================
# 3. TRAIN / TEST DATA
# ============================================================

X_train = train_df[selected_features]
y_train = train_df['Cases']

X_test = test_df[selected_features]
y_test = test_df['Cases']


# ============================================================
# 4. PARAMETER GRID
# ============================================================

param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.05, 0.1, 0.5],
    'subsample': [0.5, 1.0],
    'colsample_bytree': [0.5, 1.0]
}


# ============================================================
# 5. XGBOOST MODEL
# ============================================================

xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 6. GRID SEARCH
# ============================================================

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)


# ============================================================
# 7. GRID SEARCH RESULTS
# ============================================================

results_df = pd.DataFrame(grid.cv_results_)

print("\nTotal parameter combinations:")
print(len(results_df))


# ============================================================
# 8. FUNCTION FOR MAPE AND MPE
# ============================================================

def calculate_mape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove zero observed values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


def calculate_mpe(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove zero observed values
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 9. CALCULATE TRAIN AND TEST METRICS
#    FOR EVERY PARAMETER COMBINATION
# ============================================================

metrics_list = []

for i, params in enumerate(results_df['params']):

    print(
        f"Processing combination {i + 1} "
        f"of {len(results_df)}"
    )

    # Create model using current parameters
    model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        **params
    )

    # Fit model
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # TRAIN PREDICTIONS
    # --------------------------------------------------------

    y_train_pred = model.predict(X_train)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, y_train_pred)
    )

    train_mae = mean_absolute_error(
        y_train,
        y_train_pred
    )

    train_mape = calculate_mape(
        y_train,
        y_train_pred
    )

    train_mpe = calculate_mpe(
        y_train,
        y_train_pred
    )

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )


    # --------------------------------------------------------
    # TEST PREDICTIONS
    # --------------------------------------------------------

    y_test_pred = model.predict(X_test)

    test_rmse = np.sqrt(
        mean_squared_error(y_test, y_test_pred)
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )

    test_mape = calculate_mape(
        y_test,
        y_test_pred
    )

    test_mpe = calculate_mpe(
        y_test,
        y_test_pred
    )

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )


    # --------------------------------------------------------
    # SAVE METRICS
    # --------------------------------------------------------

    metrics_list.append({

        # Parameter values
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'learning_rate': params['learning_rate'],
        'subsample': params['subsample'],
        'colsample_bytree': params['colsample_bytree'],

        # Train metrics
        'Train_RMSE': train_rmse,
        'Train_MAE': train_mae,
        'Train_MAPE (%)': train_mape,
        'Train_MPE (%)': train_mpe,
        'Train_R2': train_r2,

        # Test metrics
        'Test_RMSE': test_rmse,
        'Test_MAE': test_mae,
        'Test_MAPE (%)': test_mape,
        'Test_MPE (%)': test_mpe,
        'Test_R2': test_r2
    })


# ============================================================
# 10. CREATE METRICS DATAFRAME
# ============================================================

metrics_df = pd.DataFrame(metrics_list)


# ============================================================
# 11. ADD GRIDSEARCH CV RESULTS
# ============================================================

metrics_df['CV_RMSE'] = (
    -results_df['mean_test_score']
)

metrics_df['CV_RMSE_SD'] = (
    results_df['std_test_score']
)


# ============================================================
# 12. IDENTIFY BEST PARAMETER BASED ON CV RMSE
# ============================================================

metrics_df['Best_CV_Model'] = False

best_index = (
    metrics_df['CV_RMSE']
    .idxmin()
)

metrics_df.loc[
    best_index,
    'Best_CV_Model'
] = True


# ============================================================
# 13. SORT BY TEST RMSE
# ============================================================

metrics_df = metrics_df.sort_values(
    by='Test_RMSE',
    ascending=True
).reset_index(drop=True)


# ============================================================
# 14. SAVE ALL RESULTS TO EXCEL
# ============================================================

output_file = (
    r"C:\Users\HPC\Desktop\DR.Sreelakshmi"
    r"\ML Revision\XGBoost\xgboost_M3_all_combinations.xlsx"
)

metrics_df.to_excel(
    output_file,
    index=False,
    sheet_name='All_Results'
)


# ============================================================
# 15. PRINT RESULTS
# ============================================================

print("\n==========================================")
print("ALL RESULTS SAVED SUCCESSFULLY")
print("==========================================")

print(f"\nFile saved at:")
print(output_file)

print("\nBest parameter combination based on CV RMSE:")
print(
    metrics_df[
        metrics_df['Best_CV_Model'] == True
    ]
)

print("\nTop 10 combinations based on Test RMSE:")
print(
    metrics_df.head(10)
)

Train shape: (4060, 44)
Test shape: (1015, 44)
Fitting 5 folds for each of 240 candidates, totalling 1200 fits

Total parameter combinations:
240
Processing combination 1 of 240
Processing combination 2 of 240
Processing combination 3 of 240
Processing combination 4 of 240
Processing combination 5 of 240
Processing combination 6 of 240
Processing combination 7 of 240
Processing combination 8 of 240
Processing combination 9 of 240
Processing combination 10 of 240
Processing combination 11 of 240
Processing combination 12 of 240
Processing combination 13 of 240
Processing combination 14 of 240
Processing combination 15 of 240
Processing combination 16 of 240
Processing combination 17 of 240
Processing combination 18 of 240
Processing combination 19 of 240
Processing combination 20 of 240
Processing combination 21 of 240
Processing combination 22 of 240
Processing combination 23 of 240
Processing combination 24 of 240
Processing combination 25 of 240
Processing combination 26 of 240
Proce

In [8]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. LOAD DATA
# ============================================================

train_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\train_M3.csv"
)

test_df = pd.read_csv(
    r"C:\Users\HPC\Downloads\New_ML\train_test_data\test_M3.csv"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

selected_features = [
    'HI', 'BI', 'CI', 'HI_lag1', 'HI_lag2', 'HI_lag3', 'BI_lag1', 'BI_lag2',
       'BI_lag3', 'CI_lag1', 'CI_lag2', 'CI_lag3',
    'Mean_temp', 'Max_temp', 'Min_temp', 'Diurnal_temp', 'Rainfall_mean',
       'Relative_humidity', 'Population_density', 'Mean_temp_lag1',
       'Mean_temp_lag2', 'Mean_temp_lag3', 'Max_temp_lag1', 'Max_temp_lag2',
       'Max_temp_lag3', 'Min_temp_lag1', 'Min_temp_lag2', 'Min_temp_lag3',
       'Diurnal_temp_lag1', 'Diurnal_temp_lag2', 'Diurnal_temp_lag3',
       'Rainfall_mean_lag1', 'Rainfall_mean_lag2', 'Rainfall_mean_lag3',
       'Relative_humidity_lag1', 'Relative_humidity_lag2',
       'Relative_humidity_lag3', 'Cases_lag1'
]


# ============================================================
# 3. TRAIN / TEST DATA
# ============================================================

X_train = train_df[selected_features]
y_train = train_df['Cases']

X_test = test_df[selected_features]
y_test = test_df['Cases']


# ============================================================
# 4. SELECTED XGBOOST PARAMETERS
# ============================================================

params = {
    'colsample_bytree': 1,
    'learning_rate': 0.05,
    'max_depth': 7,
    'n_estimators': 200,
    'subsample': 1
}


# ============================================================
# 5. BUILD XGBOOST MODEL
# ============================================================

model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    **params
)


# ============================================================
# 6. FIT MODEL
# ============================================================

model.fit(X_train, y_train)


# ============================================================
# 7. PREDICTIONS
# ============================================================

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


# ============================================================
# 8. MAPE FUNCTION
# ============================================================

def calculate_mape(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exclude observed = 0
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 9. MPE FUNCTION
# ============================================================

def calculate_mpe(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Exclude observed = 0
    mask = y_true != 0

    if np.sum(mask) == 0:
        return np.nan

    return np.mean(
        (
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    ) * 100


# ============================================================
# 10. TRAIN METRICS
# ============================================================

train_rmse = np.sqrt(
    mean_squared_error(y_train, y_train_pred)
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)

train_mape = calculate_mape(
    y_train,
    y_train_pred
)

train_mpe = calculate_mpe(
    y_train,
    y_train_pred
)

train_r2 = r2_score(
    y_train,
    y_train_pred
)


# ============================================================
# 11. TEST METRICS
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_mape = calculate_mape(
    y_test,
    y_test_pred
)

test_mpe = calculate_mpe(
    y_test,
    y_test_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)


# ============================================================
# 12. PRINT SELECTED PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("SELECTED XGBOOST M3 PARAMETERS")
print("=" * 60)

for key, value in params.items():
    print(f"{key}: {value}")

print("random_state: 42")


# ============================================================
# 13. PRINT TRAIN RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MAPE : {train_mape:.4f}%")
print(f"MPE  : {train_mpe:.4f}%")
print(f"R²   : {train_r2:.4f}")


# ============================================================
# 14. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MAPE : {test_mape:.4f}%")
print(f"MPE  : {test_mpe:.4f}%")
print(f"R²   : {test_r2:.4f}")

print("\n" + "=" * 60)
print("MODEL RUN COMPLETED")
print("=" * 60)

Train shape: (4060, 44)
Test shape: (1015, 44)

SELECTED XGBOOST M3 PARAMETERS
colsample_bytree: 1
learning_rate: 0.05
max_depth: 7
n_estimators: 200
subsample: 1
random_state: 42

TRAINING RESULTS
RMSE : 0.7770
MAE  : 0.4279
MAPE : 35.5189%
MPE  : 23.3845%
R²   : 0.9538

TEST RESULTS
RMSE : 4.0818
MAE  : 1.4674
MAPE : 84.3740%
MPE  : 2.2381%
R²   : 0.4614

MODEL RUN COMPLETED
